# Evaluating Product Behaviour - Diagnosing the System

Creating a behaviour set to evaluate the behaviour of the product, the CLinical NER pipeline. The goal is to answer the question:
    Does the system behave acceptably in the real world?

 Why not use the test set? Because the test set 

In [14]:
import pandas as pd
import json
from typing import List
from enum import Enum
import datetime, os
from dataclasses import dataclass  
from config import settings
from gcp_utils import download_from_gcs

# GENERATE BEHAVIOUR SET

## Template Bank

In [ ]:
TEMPLATE_BANK = {
    "Core symptom mention (clean baseline)": [
        "Patient reports {SYMPTOM_ENTITY}.",
        "The main complaint today is {SYMPTOM_ENTITY}.",
        "Complains of {SYMPTOM_ENTITY} since yesterday.",
        "Experiencing {SYMPTOM_ENTITY} intermittently.",
        "Presents with {SYMPTOM_ENTITY}."
    ],
    "Temporal + progression (very important clinically)": [
        "Patient reports {SYMPTOM_ENTITY} that started two days ago.",
        "Complains of {SYMPTOM_ENTITY}, which has been worsening over the past week.",
        "Describes {SYMPTOM_ENTITY} that began suddenly this morning.",
        "Has had {SYMPTOM_ENTITY} on and off for several months.",
        "Notes {SYMPTOM_ENTITY} with gradual improvement."
    ],
    "Negation & uncertainty (classic failure mode)": [
        "Denies {SYMPTOM_ENTITY}.",
        "No evidence of {SYMPTOM_ENTITY}.",
        "Patient is unsure whether the {SYMPTOM_ENTITY} is related.",
        "Does not currently have {SYMPTOM_ENTITY}, but had it previously.",
        "Reports concern about {SYMPTOM_ENTITY}, though symptoms are minimal."
    ],
    "Multiple symptoms in one sentence (boundary stress test)": [
        "Reports {SYMPTOM_ENTITY}, nausea, and fatigue.",
        "Complains of headache, dizziness, and {SYMPTOM_ENTITY}.",
        "Presents with fever, cough, and {SYMPTOM_ENTITY} since last night.",
        "Endorses {SYMPTOM_ENTITY} along with shortness of breath.",
        "Notes muscle pain and {SYMPTOM_ENTITY} after exertion."
    ],
    "Long, realistic clinical sentences (THIS IS GOLD 🥇)": [
        "The patient reports {SYMPTOM_ENTITY} that started approximately three days ago after physical exertion, associated with mild fatigue but no fever or chills.",
        "Over the past week, the patient has experienced intermittent {SYMPTOM_ENTITY}, which tends to worsen in the evenings and partially improves with rest.",
        "Patient describes persistent {SYMPTOM_ENTITY} without clear triggers, denies chest pain, nausea, or recent infections.",
        "Since the last visit, the patient notes worsening {SYMPTOM_ENTITY}, especially when walking long distances, but denies shortness of breath.",
        "The patient presents today due to concerns about {SYMPTOM_ENTITY}, which has been affecting daily activities despite over-the-counter medications."
    ],
    "Attribution & causal language (often tricky)": [
        "Patient believes the {SYMPTOM_ENTITY} may be related to recent medication changes.",
        "Attributes {SYMPTOM_ENTITY} to increased stress at work.",
        "Reports {SYMPTOM_ENTITY} following recent illness.",
        "Notes onset of {SYMPTOM_ENTITY} after starting a new exercise routine.",
        "Suspects {SYMPTOM_ENTITY} is due to poor sleep."
    ],
    "“Should NOT extract” edge cases (behavioral guardrails)": [
        "Family history notable for {SYMPTOM_ENTITY}.",
        "Educated the patient about warning signs such as {SYMPTOM_ENTITY}.",
        "Discussed possible future symptoms including {SYMPTOM_ENTITY}.",
        "No current complaints, but reviewed symptoms like {SYMPTOM_ENTITY}.",
        "Screening questionnaire includes {SYMPTOM_ENTITY}."
    ]
}

# Save template bank

# with open("behavioural_template_bank.json", "w") as f:
#     json.dump(TEMPLATE_BANK, f, ensure_ascii=False, indent=1)

## Generate the Behaviour Set

In [33]:
# Load symptoms list 
symptoms_List = pd.read_csv("base_symptom_dict.csv")['prefLabel']
print("Total number of symptoms:" , len(symptoms_List))
print("Total number of UNIQUE symptoms:" , len(set(symptoms_List)))

# Shuffle the symptoms list so we can select without replacement
symptoms_shuffled = symptoms_List.sample(frac=1, random_state=42).reset_index(drop=True)
symptom_idx = 0  # Pointer to track our position in the list

# load template bank
with open("behavioral_template_bank.json", "r") as f:
    template = json.load(f)

# Create the behavioral set!
populated_templates = {}
for k, templates in template.items():
    samples = []
    print(f"Generating examples for template: {k}")
    for temp in templates:
        # Count how many {SYMPTOM_ENTITY} placeholders are in this template
        num_placeholders = temp.count("{SYMPTOM_ENTITY}")
        
        # Pick the required number of symptoms without replacement
        chosen_symptoms = []
        for _ in range(num_placeholders):
            if symptom_idx >= len(symptoms_shuffled):
                print("Ran out of unique symptoms! Restarting from the beginning.")
                symptom_idx = 0  # RESTART in case we run out of samples, defensive programming
            
            chosen_symptom = symptoms_shuffled.iloc[symptom_idx]
            symptom_idx += 1
            chosen_symptoms.append(chosen_symptom)
        
        # Replace each placeholder with a different symptom
        example = temp
        for symptom in chosen_symptoms:
            example = example.replace("{SYMPTOM_ENTITY}", symptom, 1)  # Replace only the first occurrence
        
        # 'chosen_symptoms' is already a list of strings, so it is JSON serializable
        samples.append({"example": example, "entities": chosen_symptoms})
    populated_templates[k] = samples


Total number of symptoms: 893
Total number of UNIQUE symptoms: 893
Generating examples for template: Core symptom mention (clean baseline)
Generating examples for template: Temporal + progression (very important clinically)
Generating examples for template: Negation & uncertainty (classic failure mode)
Generating examples for template: Multiple symptoms in one sentence (boundary stress test)
Generating examples for template: Long, realistic clinical sentences (THIS IS GOLD 🥇)
Generating examples for template: Attribution & causal language (often tricky)
Generating examples for template: “Should NOT extract” edge cases (behavioral guardrails)


In [ ]:
# BEHAVIORAL SET FOR V01
# with open("v01/behavioural_set.json", "w") as f:
#     json.dump(populated_templates, f, indent=1)